In [6]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import json

In [7]:
load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")


In [8]:
CHAT_FILE = "chat_history.json"
MAX_HISTORY = 10  # Keeps the last 10 messages (5 user inputs + 5 AI replies)

# ---- STEP 1: LOAD EXISTING HISTORY OR START FRESH ----
if os.path.exists(CHAT_FILE):
    with open(CHAT_FILE, "r") as f:
        messages = json.load(f)
    print("--- Loaded past conversation history ---")
else:
    # Initialize with the permanent system rules
    messages = [{"role": "system", "content": "You are a helpful assistant."}]
    print("--- Started a brand new conversation ---")

print("Type 'quit' to exit and save.")

--- Started a brand new conversation ---
Type 'quit' to exit and save.


In [9]:
# ---- STEP 2: RUN THE CONVERSATION LOOP ----
openai = OpenAI(api_key=api_key)
while True:
    user_input = input("\nYou: ")
    if user_input.lower() == 'quit':
        break
    
    # Append the new user question
    messages.append({"role": "user", "content": user_input})
    
    # ---- STEP 3: TRIM HISTORY TO SAVE TOKENS ----
    # If history is too long, keep the system prompt [0] and the last X messages
    if len(messages) > MAX_HISTORY + 1:
        messages = [messages[0]] + messages[-MAX_HISTORY:]
        
    try:
        # Send the managed list to OpenAI
        response = openai.chat.completions.create(
            model="gpt-4o-mini",  # Highly recommended for quick chatbot testing to save costs
            messages=messages
        )
        
        # FIX: Added [0] index to choices
        reply = response.choices[0].message.content
        print(f"\nAI: {reply}")
        
        # Append the AI's reply to the history
        messages.append({"role": "assistant", "content": reply})
        
    except Exception as e:
        print(f"\nAn error occurred: {e}")
        break

# ---- STEP 4: SAVE HISTORY TO FILE ON EXIT ----
with open(CHAT_FILE, "w") as f:
    # FIX: Changed json.json.dump to json.dump
    json.dump(messages, f, indent=4)

print("\n--- Conversation saved to chat_history.json. Goodbye! ---")



AI: Hello! How can I assist you today?

AI: The value of pi (π) is approximately 3.14159. It is an irrational number, which means it has an infinite number of decimal places and cannot be expressed as a simple fraction. Pi is defined as the ratio of the circumference of a circle to its diameter. If you need more specific values or information about pi, feel free to ask!

--- Conversation saved to chat_history.json. Goodbye! ---
